# Lire CSV & Parquet depuis MinIO, agréger et sauvegarder

Ce notebook vous montre comment :

1. **Lire des fichiers CSV et Parquet directement depuis MinIO** — sans les télécharger sur votre ordinateur
2. **Les charger dans des DataFrames pandas** pour les analyser
3. **Effectuer des agrégations** (regroupement, somme, moyenne, etc.)
4. **Sauvegarder les résultats en Parquet** dans un autre bucket MinIO

## Comment ça marche sans télécharger ?

On utilise la bibliothèque **s3fs**, qui permet à pandas de communiquer avec tout stockage compatible S3 (comme MinIO)
comme s'il s'agissait d'un système de fichiers local. Les données sont **transmises en continu** en mémoire — aucun fichier temporaire sur le disque.

## Prérequis

- Packages Python : `pandas`, `s3fs`, `pyarrow` (pour le support Parquet)
- Variables d'environnement pour la connexion MinIO :
  - `MINIO_ENDPOINT` — adresse du serveur (ex. `minio.example.com:9000`)
  - `MINIO_ACCESS_KEY` — votre clé d'accès
  - `MINIO_SECRET_KEY` — votre clé secrète
  - `MINIO_SECURE` — optionnel, `true` pour HTTPS (par défaut `false`)
- Un bucket source contenant des fichiers CSV et/ou Parquet
- Un bucket de destination où les résultats seront sauvegardés (il doit déjà exister)

## Étape 1 — Installer les packages nécessaires

- `pandas` — la bibliothèque principale d'analyse de données
- `s3fs` — permet à pandas de lire/écrire directement depuis un stockage compatible S3
- `pyarrow` — le moteur qui lit et écrit les fichiers Parquet

In [ ]:
# Installer les bibliothèques nécessaires
!pip install pandas s3fs pyarrow --quiet

## Étape 2 — Importer les bibliothèques et lire les variables d'environnement

In [ ]:
import os
import pandas as pd  # Bibliothèque principale pour les données tabulaires
import s3fs           # Permet à pandas de lire/écrire vers un stockage S3

# --- Lire les paramètres de connexion depuis les variables d'environnement ---
# Les variables d'environnement gardent les secrets EN DEHORS de votre code.

endpoint   = os.environ["MINIO_ENDPOINT"]    # ex. "minio.example.com:9000"
access_key = os.environ["MINIO_ACCESS_KEY"]   # votre clé d'accès
secret_key = os.environ["MINIO_SECRET_KEY"]   # votre clé secrète
secure     = os.environ.get("MINIO_SECURE", "false").lower() == "true"

print(f"Point d'accès MinIO : {endpoint}")
print(f"Sécurisé (HTTPS)    : {secure}")

## Étape 3 — Créer la connexion au système de fichiers S3

L'objet `s3fs.S3FileSystem` est un « système de fichiers virtuel » qui pointe vers votre serveur MinIO.
Une fois créé, pandas peut l'utiliser pour ouvrir des fichiers sur MinIO aussi facilement que des fichiers locaux.

In [ ]:
# Construire l'URL du point d'accès attendu par s3fs
protocol = "https" if secure else "http"
endpoint_url = f"{protocol}://{endpoint}"

# Créer l'objet système de fichiers compatible S3
fs = s3fs.S3FileSystem(
    key=access_key,              # clé d'accès (comme un nom d'utilisateur)
    secret=secret_key,           # clé secrète (comme un mot de passe)
    endpoint_url=endpoint_url,   # où se trouve MinIO
    use_ssl=secure,              # correspondre au paramètre HTTPS
)

print(f"Connecté à {endpoint_url}")

## Étape 4 — Définir les buckets source et destination

Modifiez ces variables selon votre configuration :
- `SOURCE_BUCKET` — le bucket contenant vos fichiers CSV/Parquet bruts
- `DEST_BUCKET` — le bucket où les résultats agrégés seront sauvegardés

In [ ]:
# ---- MODIFIEZ CES VALEURS selon votre configuration MinIO ----
SOURCE_BUCKET = "raw-data"       # bucket contenant les fichiers source
DEST_BUCKET   = "results"        # bucket où les résultats seront sauvegardés

# Noms des fichiers dans le bucket source
CSV_FILE     = "sales.csv"       # un fichier CSV à lire
PARQUET_FILE = "products.parquet" # un fichier Parquet à lire

## Étape 5 — Lire un fichier CSV directement depuis MinIO

On utilise `fs.open()` pour ouvrir le fichier sur MinIO comme un flux, puis on le passe à `pd.read_csv()`.
Le fichier n'est **jamais enregistré sur le disque** — il va directement dans un DataFrame en mémoire.

In [ ]:
# Construire le chemin complet : "nom-du-bucket/nom-du-fichier"
csv_path = f"{SOURCE_BUCKET}/{CSV_FILE}"

# Ouvrir le fichier sur MinIO et le lire dans un DataFrame
# 'rb' signifie « lecture en mode binaire » — requis pour les fichiers distants
with fs.open(csv_path, "rb") as f:
    df_csv = pd.read_csv(f)

print(f"CSV chargé : {len(df_csv)} lignes, {len(df_csv.columns)} colonnes")
df_csv.head()  # afficher les 5 premières lignes

## Étape 6 — Lire un fichier Parquet directement depuis MinIO

Même approche — on ouvre le fichier distant et on le lit avec `pd.read_parquet()`.
Parquet est un format en colonnes : il est plus rapide et plus compact que le CSV pour les gros jeux de données.

In [ ]:
parquet_path = f"{SOURCE_BUCKET}/{PARQUET_FILE}"

with fs.open(parquet_path, "rb") as f:
    df_parquet = pd.read_parquet(f)

print(f"Parquet chargé : {len(df_parquet)} lignes, {len(df_parquet.columns)} colonnes")
df_parquet.head()

## Étape 7 — Effectuer des agrégations

Maintenant que les données sont chargées, faisons quelques analyses.
Voici des exemples courants d'agrégation — **adaptez les noms de colonnes à vos données**.

### Qu'est-ce qu'une agrégation ?

Une agrégation **résume plusieurs lignes en moins de lignes** par regroupement.
Par exemple : « total des ventes par catégorie de produit » regroupe toutes les lignes par catégorie
et additionne la colonne des ventes.

In [ ]:
# =============================================
# Exemple : agréger les données CSV
# Adaptez les noms de colonnes à VOS données !
# =============================================

# Supposons que df_csv a les colonnes : "category", "amount", "quantity"
#
# groupby("category") = regrouper les lignes qui partagent la même catégorie
# .agg(...)           = calculer des statistiques résumées par groupe

agg_csv = df_csv.groupby("category").agg(
    total_amount=("amount", "sum"),      # somme des montants par catégorie
    mean_amount=("amount", "mean"),       # montant moyen par catégorie
    total_quantity=("quantity", "sum"),    # somme des quantités par catégorie
    row_count=("amount", "count"),         # nombre de lignes par catégorie
).reset_index()  # remettre les étiquettes de groupe en colonne normale

print(f"CSV agrégé : {len(agg_csv)} lignes")
agg_csv

In [ ]:
# =============================================
# Exemple : agréger les données Parquet
# Adaptez les noms de colonnes à VOS données !
# =============================================

# Supposons que df_parquet a les colonnes : "region", "price", "units_sold"

agg_parquet = df_parquet.groupby("region").agg(
    total_revenue=("price", "sum"),       # revenu total par région
    avg_price=("price", "mean"),           # prix moyen par région
    total_units=("units_sold", "sum"),     # unités vendues par région
).reset_index()

print(f"Parquet agrégé : {len(agg_parquet)} lignes")
agg_parquet

## Étape 8 — Sauvegarder les résultats en Parquet dans un autre bucket MinIO

On écrit chaque DataFrame agrégé dans MinIO au **format Parquet**.
Encore une fois, pas de fichier temporaire sur le disque — les données vont directement de la mémoire vers MinIO.

In [ ]:
# Sauvegarder les résultats agrégés du CSV
output_path_csv = f"{DEST_BUCKET}/agg_sales.parquet"

with fs.open(output_path_csv, "wb") as f:   # 'wb' = écriture en mode binaire
    agg_csv.to_parquet(f, index=False)        # index=False évite de sauvegarder les numéros de lignes

print(f"Sauvegardé : {output_path_csv}")

# Sauvegarder les résultats agrégés du Parquet
output_path_parquet = f"{DEST_BUCKET}/agg_products.parquet"

with fs.open(output_path_parquet, "wb") as f:
    agg_parquet.to_parquet(f, index=False)

print(f"Sauvegardé : {output_path_parquet}")

## Étape 9 — Vérifier que les fichiers ont été sauvegardés

Listons le contenu du bucket de destination pour confirmer que nos fichiers sont bien là.

In [ ]:
# Lister tous les fichiers dans le bucket de destination
files = fs.ls(DEST_BUCKET, detail=True)

print(f"Fichiers dans '{DEST_BUCKET}' :\n")
for f in files:
    size_kb = f["size"] / 1024
    print(f"  {f['Key']:<50} {size_kb:>8.1f} Ko")

## Résumé

Dans ce notebook, vous avez appris à :

1. **Se connecter à MinIO** avec `s3fs` et des variables d'environnement
2. **Lire des fichiers CSV** directement depuis MinIO dans un DataFrame (sans téléchargement)
3. **Lire des fichiers Parquet** directement depuis MinIO dans un DataFrame (sans téléchargement)
4. **Agréger les données** avec `groupby()` et `.agg()`
5. **Sauvegarder les résultats en Parquet** dans un autre bucket MinIO (sans fichier temporaire)

### Bibliothèques clés

| Bibliothèque | Rôle                                                  |
|-------------- |-------------------------------------------------------|
| `pandas`      | Charger, transformer et analyser des données tabulaires |
| `s3fs`        | Lire/écrire vers un stockage compatible S3 (MinIO)    |
| `pyarrow`     | Lire et écrire des fichiers Parquet                   |